# Redfield Relaxation Test

## What is tested

`RedfieldRelaxationChannel` builds a relaxation superoperator from coupling operators and a
spectral density. This notebook checks that, for a constant spectral density, Redfield
relaxation agrees with the Lindblad formalism, and that Redfield contexts work correctly with
the secular approximation and with the context operations `@`, `+` and `mars.concat`.

## List of tests

**Simple tests** (`run_tests_simple`)

1. `test_lindblad_dissipator_trace`: dissipators built by `lindblad_dissipator_from_operator` preserve the trace.
2. `test_redfield_vs_lindblad_non_secular_eigen`: non-secular Redfield and Lindblad agree in the eigenbasis.
3. `test_redfield_vs_lindblad_secular_eigen`: secular Redfield and Lindblad superoperators are identical in the eigenbasis.
4. `test_redfield_vs_lindblad_non_secular_zfs`: non-secular Redfield and Lindblad agree in the ZFS basis.

**Difficult tests** (`run_tests_difficult`)

5. `test_secular_equals_masked_non_secular`: the secular superoperator equals the non-secular one with the non-secular terms removed.
6. `test_secular_populations_at_high_field`: secular and non-secular population blocks agree at high field.
7. `test_redfield_multiplication`: the context `@` gives the Liouville-space Kronecker sum.
8. `test_redfield_concatenation`: `mars.concat` gives a block-diagonal structure.
9. `test_redfield_summation`: the context `+` adds superoperators and rates.

In tests 2 and 4 Lindblad contexts are built in two ways: from a precomputed superoperator
and through `LindbladRelaxationChannel`.

## How to run

Run all cells. The last two cells call `run_tests_simple()` and `run_tests_difficult()`.
Each test prints one line when it passes and raises `AssertionError` when it fails.

In [1]:
import os
import sys
import typing as tp

import torch

# Makes MaRs importable when the notebook is run from the repository without installation
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

import mars
from mars import spin_model
from mars.population import LindbladRelaxationChannel, RedfieldRelaxationChannel, transform
from mars.population.contexts import Context

dtype = torch.float64
complex_dtype = torch.complex128

FIELD = 0.1  # T
TEMPERATURE = 300.0  # K
NUM_OPERATORS = 3  # Number of random coupling operators in every test
SEED = 42

## Helper functions

### Samples and operators

In [2]:
def create_sample(g_values: tuple, zfs_values: tuple) -> spin_model.SolidSample:
    """Create a powder sample of an S = 1 system with the given g-tensor and (D, E) in Hz."""
    system = spin_model.SpinSystem(
        electrons=[1.0],
        g_tensors=[spin_model.Interaction(g_values, dtype=dtype)],
        electron_electron=[(0, 0, spin_model.DEInteraction(zfs_values, dtype=dtype))],
        dtype=dtype,
    )
    return spin_model.SolidSample(base_spin_system=system, gauss=0.001, lorentz=0.001, mesh=(4, 4), dtype=dtype)


def create_samples() -> tuple[spin_model.SolidSample, spin_model.SolidSample]:
    """Create the two different samples used by the tests."""
    return create_sample((2.02, 2.04, 2.06), (200e6, 50e6)), create_sample((2.02, 2.14, 2.16), (200e6, 70e6))


def get_eigen_system(sample: spin_model.SolidSample, field: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Return the eigenvalues and eigenvectors of the sample Hamiltonian at the given field."""
    F, _, _, Gz = sample.get_hamiltonian_terms()
    hamiltonian = F + Gz * torch.tensor([field], dtype=dtype)
    return torch.linalg.eigh(hamiltonian.unsqueeze(-3))


def create_operators(sample: spin_model.SolidSample, number: int = NUM_OPERATORS) -> list[torch.Tensor]:
    """Create random Hermitian coupling operators for the sample."""
    dim = sample.spin_system_dim
    operators = []
    for _ in range(number):
        operator = torch.randn((dim, dim), dtype=complex_dtype)
        operators.append((operator + operator.transpose(-2, -1).conj()) / 2)
    return operators

### Spectral density and contexts

In [3]:
def spectral_density(
    omega_rad_s: torch.Tensor,
    temperature: torch.Tensor,
    random_off_diagonal: bool = False,
    seed: tp.Optional[int] = None,
) -> torch.Tensor:
    """Return a spectral density with a unit diagonal; off-diagonal values are 1 or random positive numbers."""
    if random_off_diagonal and seed is not None:
        torch.manual_seed(seed)
    density = torch.randn_like(omega_rad_s) if random_off_diagonal else torch.ones_like(omega_rad_s)
    density = density.abs()
    dim = omega_rad_s.shape[-1]
    density[..., torch.arange(dim), torch.arange(dim)] = 1.0
    return density


def create_redfield_context(
    sample,
    operators: list[torch.Tensor],
    basis: str = "zfs",
    enforce_secularity: bool = True,
    density_function: tp.Callable = spectral_density,
) -> Context:
    """Create a context with a Redfield relaxation channel and no thermal correction."""
    channel = RedfieldRelaxationChannel(
        operator_components=[(operator, None) for operator in operators],
        spectral_density_func=density_function,
        thermal_balance_mode="skip",
    )
    return Context(
        basis=basis,
        sample=sample,
        init_populations=[0.5, 0.35, 0.15][: sample.spin_system_dim],
        relaxation_coupling_channels=[channel],
        enforce_secularity=enforce_secularity,
        dtype=dtype,
    )


def create_lindblad_superop_context(
    sample, operators: list[torch.Tensor], basis: str = "zfs", enforce_secularity: bool = True
) -> Context:
    """Create a context from a precomputed Lindblad superoperator."""
    superop = sum(transform.Liouvilleator().lindblad_dissipator_from_operator(operator) for operator in operators)
    return Context(
        basis=basis,
        sample=sample,
        init_populations=[0.5, 0.35, 0.15][: sample.spin_system_dim],
        relaxation_superop=superop,
        enforce_secularity=enforce_secularity,
        dtype=dtype,
    )


def create_lindblad_channel_context(
    sample, operators: list[torch.Tensor], basis: str = "zfs", enforce_secularity: bool = True
) -> Context:
    """Create a context through the LindbladRelaxationChannel interface."""
    channel = LindbladRelaxationChannel(
        operator_components=[(operator, None) for operator in operators],
        thermal_balance_mode="skip",
    )
    return Context(
        basis=basis,
        sample=sample,
        init_populations=[0.5, 0.35, 0.15][: sample.spin_system_dim],
        relaxation_coupling_channels=[channel],
        enforce_secularity=enforce_secularity,
        dtype=dtype,
    )

### Shared calculations

In [4]:
def population_indices(dim: int, device=None) -> torch.Tensor:
    """Return the positions of the populations rho_ii in the vectorized density matrix."""
    return torch.arange(dim, device=device) * (dim + 1)


def population_block(superop: torch.Tensor, dim: int) -> torch.Tensor:
    """Return the population-population block of a superoperator."""
    index = population_indices(dim, superop.device)
    return superop[..., index.unsqueeze(-1), index.unsqueeze(-2)]


def get_superops(context_a, context_b, vectors, conditions: dict, eigen_basis: bool):
    """Return the driven superoperators of two contexts, expanding the second to the batch of the first."""
    superop_a = context_a.get_transformed_driven_superop(vectors, **conditions)
    superop_b = context_b.get_transformed_driven_superop(vectors, **conditions)
    assert superop_a is not None and superop_b is not None, "Superoperators must not be None"

    # In the eigenbasis the Lindblad superoperator is not field dependent and has no batch shape
    if eigen_basis:
        superop_b = superop_b.expand(superop_a.shape)

    assert superop_a.shape == superop_b.shape, f"Shape mismatch: {superop_a.shape} vs {superop_b.shape}"
    return superop_a, superop_b

## Checks

### Redfield and Lindblad

In [5]:
def check_redfield_vs_lindblad_elements(
    context_redfield, context_lindblad, vectors, conditions: dict, eigen_basis: bool, atol: float = 1e-4
) -> None:
    """Compare population rates, coherence decay and transition rates, and check trace preservation."""
    dim = context_redfield.spin_system_dim
    superop_redfield, superop_lindblad = get_superops(
        context_redfield, context_lindblad, vectors, conditions, eigen_basis
    )

    population_redfield = population_block(superop_redfield, dim)
    population_lindblad = population_block(superop_lindblad, dim)
    assert torch.allclose(population_redfield, population_lindblad, atol=atol), "Population blocks differ"

    coherence_pairs = [(a, b) for a in range(dim) for b in range(dim) if a != b]
    coherence_index = torch.tensor(
        [a * dim + b for a, b in coherence_pairs], dtype=torch.long, device=superop_redfield.device
    )
    decay_redfield = superop_redfield[..., coherence_index, coherence_index]
    decay_lindblad = superop_lindblad[..., coherence_index, coherence_index]
    assert torch.allclose(decay_redfield, decay_lindblad, atol=atol), (
        f"Coherence decay rates differ by {(decay_redfield - decay_lindblad).abs().max().item():.2e}"
    )
    assert torch.all(decay_redfield.real <= 1e-10), "Redfield coherence decay rates must be negative"
    assert torch.all(decay_lindblad.real <= 1e-10), "Lindblad coherence decay rates must be negative"

    # A coherence must decay at least as fast as half the sum of the decay of its two populations
    population_decay = population_redfield.sum(dim=-1)
    minimum_decay = 0.5 * (
        population_decay[..., [a for a, _ in coherence_pairs]] + population_decay[..., [b for _, b in coherence_pairs]]
    )
    assert torch.all(decay_redfield.real <= -minimum_decay.real + atol), (
        "Coherence decay must be at least half the sum of the population decay rates"
    )

    rates_redfield = context_redfield.get_transformed_driven_rates(vectors, **conditions)
    rates_lindblad = context_lindblad.get_transformed_driven_rates(vectors, **conditions)
    if rates_redfield is not None and rates_lindblad is not None:
        assert torch.allclose(rates_redfield, rates_lindblad, atol=atol), "Transition rates differ"

    pop_index = population_indices(dim, superop_redfield.device)
    for name, superop in (("Redfield", superop_redfield), ("Lindblad", superop_lindblad)):
        trace_change = superop[..., :, pop_index].sum(dim=-1)
        assert torch.allclose(trace_change, torch.zeros_like(trace_change), atol=atol), (
            f"{name} superoperator does not preserve the trace"
        )


def check_redfield_equals_lindblad(
    context_redfield, context_lindblad, vectors, conditions: dict, eigen_basis: bool, atol: float = 1e-4
) -> None:
    """Check that the Redfield and Lindblad superoperators are identical element by element."""
    superop_redfield, superop_lindblad = get_superops(
        context_redfield, context_lindblad, vectors, conditions, eigen_basis
    )
    assert torch.allclose(superop_redfield, superop_lindblad, atol=atol), (
        "Redfield and Lindblad superoperators differ"
    )

### Context operations

In [6]:
def check_multiplication(context_1, context_2, vectors_1, vectors_2, conditions_1, conditions_2, atol=1e-6, rtol=1e-5):
    """Check the dimension, the Liouville-space Kronecker sum and the rates of context_1 @ context_2."""
    vectors = transform.batched_kron(vectors_1, vectors_2)
    product = context_1 @ context_2

    dim_1, dim_2 = context_1.spin_system_dim, context_2.spin_system_dim
    dim_total = product.spin_system_dim
    assert dim_total == dim_1 * dim_2, f"Dimension mismatch: {dim_1} x {dim_2} = {dim_total}"

    # Energies of independent subsystems add: E_total[a, b] = E_1[a] + E_2[b]
    energies_1, energies_2 = conditions_1["energies"], conditions_2["energies"]
    energies_total = energies_1.unsqueeze(-1) + energies_2.unsqueeze(-2)
    energies_total = energies_total.reshape(*energies_1.shape[:-1], -1)
    conditions = {**conditions_1, "energies": energies_total}

    superop_1 = context_1.get_transformed_driven_superop(vectors_1, **conditions_1)
    superop_2 = context_2.get_transformed_driven_superop(vectors_2, **conditions_2)
    superop_total = product.get_transformed_driven_superop(vectors, **conditions)

    # Expected superoperator: R_1 x I + I x R_2
    size_1, size_2 = dim_1 * dim_1, dim_2 * dim_2
    batch_shape = torch.broadcast_shapes(superop_1.shape[:-2], superop_2.shape[:-2])
    identity_1 = torch.eye(size_1, device=superop_1.device, dtype=superop_1.dtype).expand(batch_shape + (size_1, size_1))
    identity_2 = torch.eye(size_2, device=superop_2.device, dtype=superop_2.dtype).expand(batch_shape + (size_2, size_2))
    expected = transform.batched_kron(superop_1.expand(batch_shape + (size_1, size_1)), identity_2) + (
        transform.batched_kron(identity_1, superop_2.expand(batch_shape + (size_2, size_2)))
    )
    expected = transform.reshape_superoperator_tensor_to_kronecker_basis(expected, subsystem_dims=[dim_1, dim_2])
    assert torch.allclose(superop_total, expected, atol=atol, rtol=rtol), (
        f"Superoperators do not form a Kronecker sum. Max difference: {(superop_total - expected).abs().max().item():.2e}"
    )

    # The rates, with the drain on the diagonal, must match the population block
    rates_total = product.get_transformed_driven_rates(vectors, **conditions)
    index = torch.arange(dim_total, device=rates_total.device)
    rates_total[..., index, index] = -rates_total.sum(dim=-2)
    expected_rates = population_block(expected, dim_total).real
    assert torch.allclose(rates_total, expected_rates, atol=atol, rtol=rtol), (
        f"Rates do not match. Max difference: {(rates_total - expected_rates).abs().max().item():.2e}"
    )


def check_concatenation(context_1, context_2, vectors_1, vectors_2, conditions_1, conditions_2, atol=1e-6, rtol=1e-5):
    """Check the dimension and the block-diagonal structure of concatenated Redfield contexts."""
    concatenated = mars.concat([context_1, context_2])
    n, m = vectors_1.shape[-1], vectors_2.shape[-1]
    vectors = torch.zeros(*vectors_1.shape[:-2], n + m, n + m, dtype=vectors_1.dtype, device=vectors_1.device)
    vectors[..., :n, :n] = vectors_1
    vectors[..., n:, n:] = vectors_2

    dim_1, dim_2 = context_1.spin_system_dim, context_2.spin_system_dim
    assert concatenated.spin_system_dim == dim_1 + dim_2, (
        f"Dimension mismatch: {dim_1} + {dim_2} = {concatenated.spin_system_dim}"
    )

    conditions = {
        **conditions_1,
        "energies": torch.cat([conditions_1["energies"], conditions_2["energies"]], dim=-1),
    }

    superop_1 = context_1.get_transformed_driven_superop(vectors_1, **conditions_1)
    superop_2 = context_2.get_transformed_driven_superop(vectors_2, **conditions_2)
    superop_total = concatenated.get_transformed_driven_superop(vectors, **conditions)
    if superop_1 is not None and superop_2 is not None and superop_total is not None:
        block_1, block_2 = transform.reshape_direct_sum_basis_to_superoperators_list(superop_total, (dim_1, dim_2))
        assert torch.allclose(block_1, superop_1, atol=atol, rtol=rtol), (
            f"Block 1 differs. Max difference: {(block_1 - superop_1).abs().max().item():.2e}"
        )
        assert torch.allclose(block_2, superop_2, atol=atol, rtol=rtol), (
            f"Block 2 differs. Max difference: {(block_2 - superop_2).abs().max().item():.2e}"
        )

    rates_1 = context_1.get_transformed_driven_rates(vectors_1, **conditions_1)
    rates_2 = context_2.get_transformed_driven_rates(vectors_2, **conditions_2)
    rates_total = concatenated.get_transformed_driven_rates(vectors, **conditions)
    if rates_1 is not None and rates_2 is not None and rates_total is not None:
        expected = torch.zeros(*rates_1.shape[:-2], dim_1 + dim_2, dim_1 + dim_2, device=rates_1.device, dtype=rates_1.dtype)
        expected[..., :dim_1, :dim_1] = rates_1
        expected[..., dim_1:, dim_1:] = rates_2
        assert torch.allclose(rates_total, expected, atol=atol, rtol=rtol), (
            f"Rates are not block diagonal. Max difference: {(rates_total - expected).abs().max().item():.2e}"
        )


def check_summation(context_1, context_2, vectors, conditions: dict, atol: float = 1e-6, rtol: float = 1e-5) -> None:
    """Check that the dimension is kept and that superoperators and rates add up."""
    summed = context_1 + context_2

    dim_1, dim_2 = context_1.spin_system_dim, context_2.spin_system_dim
    assert dim_1 == dim_2 == summed.spin_system_dim, f"Dimension mismatch: {dim_1} + {dim_2} = {summed.spin_system_dim}"

    superop_1 = context_1.get_transformed_driven_superop(vectors, **conditions)
    superop_2 = context_2.get_transformed_driven_superop(vectors, **conditions)
    superop_sum = summed.get_transformed_driven_superop(vectors, **conditions)
    if superop_1 is not None and superop_2 is not None and superop_sum is not None:
        assert torch.allclose(superop_sum, superop_1 + superop_2, atol=atol, rtol=rtol), "Superoperators do not add up"

    rates_1 = context_1.get_transformed_driven_rates(vectors, **conditions)
    rates_2 = context_2.get_transformed_driven_rates(vectors, **conditions)
    rates_sum = summed.get_transformed_driven_rates(vectors, **conditions)
    if rates_1 is not None and rates_2 is not None and rates_sum is not None:
        assert torch.allclose(rates_sum, rates_1 + rates_2, atol=atol, rtol=rtol), "Rates do not add up"

### Secular approximation

In [7]:
def get_secular_mask(dim: int) -> torch.Tensor:
    """Return the mask of secular terms: population transfer and the diagonal of the coherences."""
    size = dim * dim
    mask = torch.eye(size, dtype=torch.bool)
    index = population_indices(dim)
    mask[index[:, None], index[None, :]] = True
    return mask


def check_secular_equals_masked_non_secular(
    context_secular, context_non_secular, vectors, conditions: dict, atol: float = 1e-6, rtol: float = 1e-5
) -> None:
    """Compare secular and non-secular contexts: rates, population blocks and the secular part of the superoperator."""
    dim = context_secular.spin_system_dim
    superop_secular = context_secular.get_transformed_driven_superop(vectors, **conditions)
    superop_non_secular = context_non_secular.get_transformed_driven_superop(vectors, **conditions)
    rates_secular = context_secular.get_transformed_driven_rates(vectors, **conditions)
    rates_non_secular = context_non_secular.get_transformed_driven_rates(vectors, **conditions)

    # Add the population drain to the diagonal of the rates
    index = torch.arange(dim, device=rates_secular.device)
    rates_secular[..., index, index] = -rates_secular.sum(dim=-2)
    rates_non_secular[..., index, index] = -rates_non_secular.sum(dim=-2)

    population_secular = population_block(superop_secular, dim)
    population_non_secular = population_block(superop_non_secular, dim)

    assert torch.allclose(population_secular.real, rates_secular.real, atol=atol, rtol=rtol), (
        "Secular population block differs from the secular rates"
    )
    assert torch.allclose(rates_non_secular.real, rates_secular.real, atol=atol, rtol=rtol), (
        "Secular and non-secular rates differ"
    )
    assert torch.allclose(population_non_secular.real, rates_non_secular.real, atol=atol, rtol=rtol), (
        "Non-secular population block differs from the non-secular rates"
    )
    assert torch.allclose(population_secular.real, population_non_secular.real, atol=atol, rtol=rtol), (
        "Secular and non-secular population blocks differ"
    )

    assert superop_secular.shape == superop_non_secular.shape, (
        f"Shape mismatch: {superop_secular.shape} vs {superop_non_secular.shape}"
    )
    masked = superop_non_secular * get_secular_mask(dim).type_as(superop_non_secular)
    assert torch.allclose(masked.imag, torch.zeros_like(masked.imag), atol=atol, rtol=rtol), (
        "The secular part must be real"
    )
    assert torch.allclose(masked, superop_secular, atol=atol, rtol=rtol), (
        "The secular superoperator differs from the masked non-secular one"
    )

## Test setup

In [8]:
def create_setup(field: float = FIELD) -> dict:
    """Create the samples and their eigenvectors and field-dependent conditions."""
    sample_1, sample_2 = create_samples()
    energies_1, vectors_1 = get_eigen_system(sample_1, field)
    energies_2, vectors_2 = get_eigen_system(sample_2, field)
    fields = torch.tensor([field], dtype=dtype).unsqueeze(-2)
    temperature = torch.tensor(TEMPERATURE, dtype=dtype)
    return {
        "sample_1": sample_1,
        "sample_2": sample_2,
        "vectors_1": vectors_1,
        "vectors_2": vectors_2,
        "conditions_1": {"fields": fields, "energies": energies_1, "temperature": temperature},
        "conditions_2": {"fields": fields, "energies": energies_2, "temperature": temperature},
    }

## Simple tests

In [9]:
def test_lindblad_dissipator_trace():
    """Check the shape and trace preservation of dissipators built from diagonal jump operators."""
    sample_1, _ = create_samples()
    dim = sample_1.spin_system_dim
    size = dim * dim

    for level in range(min(3, dim)):
        operator = torch.zeros((dim, dim), dtype=complex_dtype)
        operator[level, level] = 1.0
        superop = transform.Liouvilleator().lindblad_dissipator_from_operator(operator.unsqueeze(0))

        assert superop.shape[-2:] == (size, size), f"Operator {level}: wrong superoperator shape"
        trace_change = superop[..., :, population_indices(dim)].sum(dim=-1)
        assert torch.allclose(trace_change, torch.zeros_like(trace_change), atol=1e-5), (
            f"Operator {level}: the superoperator does not preserve the trace"
        )
    print("Lindblad dissipator test passed (correct shape, trace is preserved)")


def test_redfield_vs_lindblad_non_secular_eigen():
    """Check that non-secular Redfield and both Lindblad interfaces agree in the eigenbasis."""
    torch.manual_seed(SEED)
    setup = create_setup()
    sample = setup["sample_1"]
    operators = create_operators(sample)
    arguments = {"basis": "eigen", "enforce_secularity": False}
    redfield = create_redfield_context(sample, operators, **arguments)

    for create_lindblad in (create_lindblad_superop_context, create_lindblad_channel_context):
        lindblad = create_lindblad(sample, operators, **arguments)
        check_redfield_vs_lindblad_elements(redfield, lindblad, setup["vectors_1"], setup["conditions_1"], eigen_basis=True)
    print("Redfield vs Lindblad test passed (non-secular, eigenbasis: populations, coherences, rates, trace)")


def test_redfield_vs_lindblad_secular_eigen():
    """Check that the secular Redfield superoperator is identical to both Lindblad interfaces in the eigenbasis."""
    torch.manual_seed(SEED)
    setup = create_setup()
    sample = setup["sample_1"]
    operators = create_operators(sample)
    arguments = {"basis": "eigen", "enforce_secularity": True}
    redfield = create_redfield_context(sample, operators, **arguments)

    for create_lindblad in (create_lindblad_superop_context, create_lindblad_channel_context):
        lindblad = create_lindblad(sample, operators, **arguments)
        check_redfield_equals_lindblad(redfield, lindblad, setup["vectors_1"], setup["conditions_1"], eigen_basis=True)
    print("Redfield vs Lindblad test passed (secular, eigenbasis: the superoperators are identical)")


def test_redfield_vs_lindblad_non_secular_zfs():
    """Check that non-secular Redfield and both Lindblad interfaces agree in the ZFS basis."""
    torch.manual_seed(SEED)
    setup = create_setup()
    sample = setup["sample_1"]
    operators = create_operators(sample)
    arguments = {"basis": "zfs", "enforce_secularity": False}
    redfield = create_redfield_context(sample, operators, **arguments)

    for create_lindblad in (create_lindblad_superop_context, create_lindblad_channel_context):
        lindblad = create_lindblad(sample, operators, **arguments)
        check_redfield_vs_lindblad_elements(redfield, lindblad, setup["vectors_1"], setup["conditions_1"], eigen_basis=False)
    print("Redfield vs Lindblad test passed (non-secular, ZFS basis: populations, coherences, rates, trace)")

## Difficult tests

In [10]:
def test_secular_equals_masked_non_secular():
    """Check that secular Redfield equals non-secular Redfield with the non-secular terms removed."""
    torch.manual_seed(SEED)
    setup = create_setup()
    sample = setup["sample_1"]
    operators = create_operators(sample)

    def random_density(omega: torch.Tensor, temperature: torch.Tensor) -> torch.Tensor:
        """Return the same random spectral density at each call, because the seed is fixed inside."""
        return spectral_density(omega, temperature, random_off_diagonal=True, seed=SEED)

    conditions = {**setup["conditions_1"], "fields": torch.tensor([FIELD], dtype=dtype)}
    context_secular = create_redfield_context(sample, operators, "eigen", True, random_density)
    context_non_secular = create_redfield_context(sample, operators, "eigen", False, random_density)

    check_secular_equals_masked_non_secular(context_secular, context_non_secular, setup["vectors_1"], conditions)
    print("Secular approximation test passed (secular equals the masked non-secular superoperator)")


def test_secular_populations_at_high_field():
    """Check that secular and non-secular population blocks agree at fields of at least 0.5 T."""
    torch.manual_seed(SEED)
    sample, _ = create_samples()
    dim = sample.spin_system_dim
    operators = create_operators(sample)
    temperature = torch.tensor(TEMPERATURE, dtype=dtype)

    for field in [0.01, 0.1, 0.5, 1.0]:
        energies, vectors = get_eigen_system(sample, field)
        conditions = {"fields": torch.tensor([field], dtype=dtype), "energies": energies, "temperature": temperature}
        superop_secular = create_redfield_context(sample, operators, "eigen", True).get_transformed_driven_superop(
            vectors, **conditions
        )
        superop_non_secular = create_redfield_context(sample, operators, "eigen", False).get_transformed_driven_superop(
            vectors, **conditions
        )
        difference = (population_block(superop_secular, dim) - population_block(superop_non_secular, dim)).abs().max().item()
        if field >= 0.5:
            assert difference < 1e-4, f"At {field} T the population blocks differ by {difference:.2e}"
    print("High field secular test passed (population blocks agree at 0.5 T and above)")


def create_two_redfield_contexts(setup: dict) -> tuple[Context, Context]:
    """Create non-secular Redfield contexts for the two samples in the ZFS basis."""
    torch.manual_seed(SEED)
    return (
        create_redfield_context(setup["sample_1"], create_operators(setup["sample_1"]), "zfs", False),
        create_redfield_context(setup["sample_2"], create_operators(setup["sample_2"]), "zfs", False),
    )


def test_redfield_multiplication():
    """Check that the product of two Redfield contexts is a Liouville-space Kronecker sum."""
    setup = create_setup()
    context_1, context_2 = create_two_redfield_contexts(setup)
    check_multiplication(
        context_1, context_2, setup["vectors_1"], setup["vectors_2"], setup["conditions_1"], setup["conditions_2"]
    )
    print("Multiplication test passed (dimension, Kronecker sum of superoperators, rates)")


def test_redfield_concatenation():
    """Check that concatenated Redfield contexts form a block-diagonal structure."""
    setup = create_setup()
    context_1, context_2 = create_two_redfield_contexts(setup)
    check_concatenation(
        context_1, context_2, setup["vectors_1"], setup["vectors_2"], setup["conditions_1"], setup["conditions_2"]
    )
    print("Concatenation test passed (dimension, block-diagonal superoperators and rates)")


def test_redfield_summation():
    """Check that two summed Redfield contexts on one sample add their superoperators and rates."""
    torch.manual_seed(SEED)
    setup = create_setup()
    sample = setup["sample_1"]
    operators = create_operators(sample)
    context_1 = create_redfield_context(sample, operators, "zfs")
    context_2 = create_redfield_context(sample, operators, "zfs")

    check_summation(context_1, context_2, setup["vectors_1"], setup["conditions_1"])
    print("Summation test passed (superoperators and rates add up)")

## Run the tests

In [11]:
def run_tests_simple():
    """Run the simple tests of this notebook."""
    tests = [
        test_lindblad_dissipator_trace,
        test_redfield_vs_lindblad_non_secular_eigen,
        test_redfield_vs_lindblad_secular_eigen,
        test_redfield_vs_lindblad_non_secular_zfs,
    ]
    for test in tests:
        test()
    print(f"\nAll {len(tests)} simple Redfield relaxation tests passed.")


def run_tests_difficult():
    """Run the difficult tests of this notebook."""
    tests = [
        test_secular_equals_masked_non_secular,
        test_secular_populations_at_high_field,
        test_redfield_multiplication,
        test_redfield_concatenation,
        test_redfield_summation,
    ]
    for test in tests:
        test()
    print(f"\nAll {len(tests)} difficult Redfield relaxation tests passed.")

In [12]:
run_tests_simple()

Lindblad dissipator test passed (correct shape, trace is preserved)
Redfield vs Lindblad test passed (non-secular, eigenbasis: populations, coherences, rates, trace)
Redfield vs Lindblad test passed (secular, eigenbasis: the superoperators are identical)
Redfield vs Lindblad test passed (non-secular, ZFS basis: populations, coherences, rates, trace)

All 4 simple Redfield relaxation tests passed.


In [13]:
run_tests_difficult()

Secular approximation test passed (secular equals the masked non-secular superoperator)
High field secular test passed (population blocks agree at 0.5 T and above)
Multiplication test passed (dimension, Kronecker sum of superoperators, rates)
Concatenation test passed (dimension, block-diagonal superoperators and rates)
Summation test passed (superoperators and rates add up)

All 5 difficult Redfield relaxation tests passed.


D:\ITC\РНФ_Курганский_2024\pythonProject\MarS\mars\population\concatination.py:270: UserWarning: Contexts are combined by direct sum (block-diagonal composition). The subsystems evolve independently until transitions between them are added, e.g., mars.concat((context_1, context_2)) + context_exchange. Transitions between subsystems with different Hamiltonians are treated as incoherent population transfer; spectral (chemical) exchange is not described. This is valid only in the slow-exchange limit, when the transition rates between subsystems are much smaller than the differences between their resonance frequencies.
  warnings.warn(
